# Figure S2 — injection-dependent surface recombination

This notebook reproduces **Figure S2**. It calculates the effective surface-recombination velocity versus fixed charge for the experimental n- and p-InP doping levels over a range of excess-carrier densities. Circles identify the surface condition where $n_s=p_s$.


In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm, Normalize
from matplotlib.ticker import LogFormatterMathtext
from scipy.constants import h, c
from IPython.display import display

root = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p/'src'/'surpass').exists()), None)
if root is None:
    raise FileNotFoundError('Could not locate the SURPASS project root.')
sys.path.insert(0, str(root/'src'))

import surpass
from surpass import (
    BulkModel, BulkTransportTable, InterfaceChargeModel, InterfaceDefectModel,
    Layer, OpticalStack, SteadyState1DSolver, SurfaceBoundaryTable,
    SurfaceSRHModel, Wafer, get_optical_material, reconstruct_band_profile,
)

print('surpass version:', surpass.__version__)
print('loaded from:', Path(surpass.__file__).resolve())
if tuple(map(int, surpass.__version__.split('.')[:3])) < (0, 9, 0):
    raise ImportError('This notebook requires surpass 1.0.0 or newer.')

try:
    import scienceplots
    plt.style.use(['science', 'notebook', 'no-latex'])
except ImportError:
    plt.style.use('default')

plt.rcParams.update({
    'figure.dpi': 120, 'savefig.dpi': 300, 'font.size': 10,
    'axes.spines.top': True, 'axes.spines.right': True,
})
output_dir = Path.cwd()/'output'
output_dir.mkdir(parents=True, exist_ok=True)

## 1. Experimental inputs and model switches

Absolute PL from n- and p-InP is not compared because different incident
intensities were used. Every simulated PL map is normalized to a separately
calculated **bare InP** reference of the same wafer polarity, using
$D_{it}=10^{12}$ eV$^{-1}$ cm$^{-2}$ and $Q_f=0$. All points in the map use
the 4-nm PO$_x$/13-nm AlO$_x$ optical stack. The plotted ratio therefore
includes the change in pump coupling caused by adding the dielectric stack,
but remains an internal-PL proxy and does not include stack-dependent
luminescence extraction or detector collection.

The nominal electrostatics omits $Q_{it}$ because the charge-neutrality level
and amphoteric-state distribution of PO$_x$/InP are not independently known.
Setting `INCLUDE_INTERFACE_CHARGE=True` activates an explicitly labelled
sensitivity case; it must not be mistaken for a measured interface model.

The energy-independent capture cross sections
$\sigma_n=10^{-14}$ cm$^2$ and $\sigma_p=10^{-13}$ cm$^2$ originate from the
InP(100) surface-state fit of Adamowicz *et al.*, *Vacuum* **63**, 223--227
(2001), DOI: 10.1016/S0042-207X(01)00195-6. That work considered a chemically
polished InP surface and a U-shaped state distribution. The values are used
here only as literature-motivated starting parameters; they are not measured
capture cross sections for PO$_x$/InP or PO$_x$/AlO$_x$/InP.


In [ ]:
wavelength_nm = 514.0
thickness_um = 620.0
thickness_cm = thickness_um*1e-4
wafer_settings = {
    'n-InP': dict(wafer=Wafer('InP', donor_cm3=8.5e18, thickness_um=thickness_um),
                  intensity_w_cm2=1.0, color='tab:blue'),
    'p-InP': dict(wafer=Wafer('InP', acceptor_cm3=5.1e18, thickness_um=thickness_um),
                  intensity_w_cm2=13.0, color='tab:red'),
}

observations = pd.DataFrame([
    {'wafer':'n-InP', 'PL stack/bare':3.071, 'TRPL bare (ns)':2.55,
     'TRPL stack (ns)':3.40, 'PL after -6e12 corona/before':0.70},
    {'wafer':'p-InP', 'PL stack/bare':0.800, 'TRPL bare (ns)':1.88,
     'TRPL stack (ns)':1.79, 'PL after -6e12 corona/before':1.30},
])
display(observations)

# Adamowicz et al., Vacuum 63, 223-227 (2001), DOI:
# 10.1016/S0042-207X(01)00195-6. Their fit to a chemically polished
# InP(100) surface used sigma_n=1e-14 and sigma_p=1e-13 cm^2. These are
# literature-motivated starting values, not fitted POx/InP parameters.
DIT_MAP = 3e10
SIGMA_N = 1e-14
SIGMA_P = 1e-13
VTH_N = 4.13e7
VTH_P = 1.51e7
tau_srh = 5e-9 #np.inf
defects = InterfaceDefectModel(
    DIT_MAP, SIGMA_N, SIGMA_P, VTH_N, VTH_P,
    shape='constant', energy_points=201,
)
print(defects.source)

INCLUDE_INTERFACE_CHARGE = True
interface_charge = (InterfaceChargeModel(0.5, 0.02)
                    if INCLUDE_INTERFACE_CHARGE else None)
ENABLE_BGN = True

# Active photon-reabsorption probability used to reduce the net radiative
# carrier loss. Use 0.0 for the conservative no-recycling baseline and 0.93
# for the high-recycling sensitivity case estimated in notebook 08.
# The intrinsic radiative-emission rate used as the PL proxy remains Bnp.
PHOTON_RECYCLING_PROBABILITY = 0.93
if not 0.0 <= PHOTON_RECYCLING_PROBABILITY < 1.0:
    raise ValueError('PHOTON_RECYCLING_PROBABILITY must satisfy 0 <= p_rec < 1')

def surface_model(wafer, charge_cm2, local_defects=defects):
    return SurfaceSRHModel(
        BulkModel(wafer, enable_bgn=ENABLE_BGN), charge_cm2, local_defects,
        electrostatics='self_consistent', driving_force='excess_product',
        interface_charge=interface_charge,
    )

## 3. Injection-dependent cuts at the actual dopings — main-text candidate

For every $(Q_{ext},\Delta n_b)$ pair the surface potential is solved again
from

$$Q_{ext}+Q_{it}+Q_{sc}=0,$$

using the illuminated electron and hole quasi-Fermi levels. This is important:
the calculation does not insert illuminated carrier populations into a frozen
dark band diagram. Markers identify $n_s=p_s$ where that crossing lies inside
the plotted charge range.

In [ ]:
injection_levels = np.array([1e4, 1e6, 1e8, 1e11, 1e13, 1e15, 1e16, 1e17, 1e18])
q_cuts = np.linspace(-1.5e13, 1.5e13, 161)
cut_data = {}

def zero_crossing(x, y):
    indices = np.flatnonzero(np.signbit(y[:-1]) != np.signbit(y[1:]))
    if not indices.size:
        return np.nan
    i = indices[0]
    return float(np.interp(0, y[i:i+2], x[i:i+2]))

for label, settings in wafer_settings.items():
    curves = []
    for delta in injection_levels:
        seff, rate, ratio = [], [], []
        for charge in q_cuts:
            state = surface_model(settings['wafer'], charge).state(delta)
            seff.append(state.effective_s_cm_s)
            rate.append(state.recombination_cm2_s)
            ratio.append(np.log10(state.n_surface_cm3/state.p_surface_cm3))
        curves.append(dict(delta=delta, seff=np.array(seff), rate=np.array(rate),
                           balance=np.array(ratio),
                           q_equal=zero_crossing(q_cuts, np.array(ratio))))
    cut_data[label] = curves

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.2), sharey=True,
                         constrained_layout=True)
norm = LogNorm(injection_levels.min(), injection_levels.max())
cmap = plt.get_cmap('viridis')
for ax, (label, settings) in zip(axes, wafer_settings.items()):
    for curve in cut_data[label]:
        color = cmap(norm(curve['delta']))
        displayed = np.clip(curve['seff'], 1e-1, 1e8)
        ax.semilogy(q_cuts/1e12, displayed, color=color, lw=1.25)
        if np.isfinite(curve['q_equal']):
            y_equal = np.clip(np.interp(curve['q_equal'], q_cuts, curve['seff']),
                              1e-1, 1e8)
            ax.plot(curve['q_equal']/1e12, y_equal, 'o', color=color,
                    mec='black', mew=.45, ms=4.5)
    ax.set_title(label + r'; circles: $n_s=p_s$')
    ax.set_xlabel(r'$Q_{ext}/q$ ($10^{12}$ cm$^{-2}$)')
    ax.set_ylabel(r'$S_{eff}=U_s/\Delta n_b$ (cm s$^{-1}$)')
    ax.set_ylim(1e-1,3e8)
    ax.text(
        .02, .94,
        r'$S_{\mathrm{eff}}\geq10^8$: transport-limited',
        transform=ax.transAxes,
        fontsize=10,
        va='top',
        bbox=dict(
            facecolor='white',
            alpha=.80,
            edgecolor='0.7',
            linewidth=0.6,
            boxstyle='round,pad=.25',
        ),
    )

# After constructing both panels:
axes[0].texts[-1].set_position((0.98, 0.94))
axes[0].texts[-1].set_ha('right')

fig.colorbar(plt.cm.ScalarMappable(norm=norm, cmap=cmap), ax=axes,
             label=r'Imposed bulk injection, $\Delta n_b$ (cm$^{-3}$)')
fig.savefig(output_dir/'paper_inp_main_injection_dependent_cuts.png', bbox_inches='tight')

crossing_rows = []
for label, curves in cut_data.items():
    for curve in curves:
        crossing_rows.append({'wafer':label, 'Delta n (cm^-3)':curve['delta'],
                              'Q at ns=ps (cm^-2)':curve['q_equal'],
                              'Q at max Us (cm^-2)':q_cuts[np.argmax(curve['rate'])],
                              'Q at max Seff (cm^-2)':q_cuts[np.argmax(curve['seff'])]})
display(pd.DataFrame(crossing_rows))

### Interpretation

The balance point and the recombination maximum are related but need not
coincide. Unequal electron/hole capture cross sections skew the SRH maximum,
while $S_{eff}=U_s/\Delta n_b$ and $U_s$ are different observables. At higher
injection the quasi-Fermi levels, interface occupancy and space charge all
change. Once injection approaches the doping density, the usual dark notions
of depletion and inversion also become less sharp.

This figure is therefore a caution against assigning the corona turning point
to a dark $n_s=p_s$ condition without specifying illumination.